# LSTM Event Detection Training - Automated Dataset

## ✅ Configuration:
- **Input File**: `sensor_data_balanced.csv` (60:10:10:10:10 ratio)
- **Window Size**: 40 samples (LSTM_WINDOW_SIZE)
- **Stride**: 10 samples (LSTM_STRIDE)
- **Features**: 7 (speed_kmh, ax_g, ay_g, az_g, gx_dps, gy_dps, gz_dps)
- **Model Architecture**: 2-layer BiLSTM (128→96) + 4 dropout layers
- **Output**: `best_model.pth` (ready for deployment)

## 🎯 Training Goal:
- Achieve **diagonal confusion matrix >= 0.6** for ALL classes
- Classes: bump, left, right, stop, straight
- Balanced dataset with 60:10:10:10:10 ratio

## 📝 Steps:
1. Run all cells in order
2. Wait for training to complete (will auto-stop when optimal)
3. Check confusion matrices
4. Deploy `best_model.pth` to Raspberry Pi

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# Try to use GPU, fallback to CPU if busy/unavailable
try:
    if torch.cuda.is_available():
        # Test if GPU is actually accessible
        torch.cuda.empty_cache()
        test_tensor = torch.zeros(1).cuda()
        del test_tensor
        print("GPU device:", torch.cuda.get_device_name(0))
        device = torch.device('cuda')
    else:
        device = torch.device('cpu')
except RuntimeError as e:
    print(f"GPU unavailable ({e}), falling back to CPU")
    device = torch.device('cpu')

print("Using device:", device)


PyTorch version: 2.9.1+cpu
CUDA available: False
Using device: cpu


In [ ]:
import os

# Auto-detect balanced dataset
data_file = None
search_paths = [
    "../final/sensor_data_balanced.csv",
    "./sensor_data_balanced.csv",
    "sensor_data214_cleaned.csv"
]

for path in search_paths:
    if os.path.exists(path):
        data_file = path
        print(f"✅ Found dataset: {data_file}")
        break

if data_file is None:
    raise FileNotFoundError("No dataset found! Please ensure sensor_data_balanced.csv exists.")

data = pd.read_csv(data_file)
print("✅ Data loaded:", data.shape)
print(data.head())

# Clean labels (strip whitespace) - NO ROWS DROPPED!
if 'label' in data.columns:
    data['label'] = data['label'].astype(str).str.strip().str.lower()

print("\n=== Label distribution (ALL data, NO rows dropped) ===")
print(data['label'].value_counts().sort_index())

# Features and labels
# Convert GPS knots to km/h (MATCHING nov21.py)
# Fill NaN with 0 for GPS speed (don't drop rows!)
data['speed_kmh'] = data['gps_speed_kn'].fillna(0) * 1.852

# Fill NaN with 0 for IMU sensors (don't drop rows!)
data['ax_g'] = data['ax_g'].fillna(0)
data['ay_g'] = data['ay_g'].fillna(0)
data['az_g'] = data['az_g'].fillna(0)
data['gx_dps'] = data['gx_dps'].fillna(0)
data['gy_dps'] = data['gy_dps'].fillna(0)
data['gz_dps'] = data['gz_dps'].fillna(0)

X = data[['speed_kmh','ax_g','ay_g','az_g','gx_dps','gy_dps','gz_dps']].values
y = data['label'].values

print(f"\n✅ Total samples: {len(X)} (NO rows dropped!)")

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
print("Classes:", label_encoder.classes_)
print("Number of classes:", len(label_encoder.classes_))

# Standardize features (MATCHING nov21.py)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# CRITICAL: MUST MATCH nov21.py EXACTLY
n_time_steps = 40  # LSTM_WINDOW_SIZE in nov21.py
n_features = X_scaled.shape[1]
step_size = 10  # LSTM_STRIDE in nov21.py

X_seq, y_seq = [], []
for i in range(0, len(X_scaled) - n_time_steps, step_size):
    X_seq.append(X_scaled[i:i + n_time_steps])
    y_seq.append(y_encoded[i + n_time_steps - 1])

X_seq, y_seq = np.array(X_seq), np.array(y_seq)
print("✅ Sequence shape:", X_seq.shape, y_seq.shape)

# STRATIFIED Train/Val/Test Split (70/15/15) - ENSURES ALL CLASSES IN ALL SETS!
from sklearn.model_selection import train_test_split

# First split: 70% train, 30% temp (val+test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_seq, y_seq, test_size=0.30, random_state=42, stratify=y_seq
)

# Second split: Split temp into 50/50 (15% val, 15% test)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print("\n=== Class distribution ===")
for split_name, y_split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    counts = Counter(y_split)
    print(f"{split_name}:")
    for i, name in enumerate(label_encoder.classes_):
        print(f"  {name}: {counts.get(i, 0)}")

# Check if dataset is already balanced (ratio close to 60:10:10:10:10)
train_counts_orig = Counter(y_train)
class_names = label_encoder.classes_

# Calculate ratios
counts_list = [train_counts_orig.get(i, 0) for i in range(len(class_names))]
min_count = min([c for c in counts_list if c > 0])
ratios = [c / min_count for c in counts_list]

print(f"\n=== Dataset balance check ===")
for i, name in enumerate(class_names):
    print(f"  {name}: {counts_list[i]} samples (ratio: {ratios[i]:.1f})")

# Check if already balanced (allow some tolerance)
is_balanced = (
    max(ratios) <= 8.0 and  # Not too imbalanced
    min([c for c in counts_list if c > 0]) >= 100  # Enough samples per class
)

if is_balanced:
    print("\n✅ Dataset appears balanced - minimal augmentation needed")
    # Use minimal downsampling for majority class only
    max_allowed = min_count * 7  # Allow up to 7x ratio
    keep_indices = []
    for i in range(len(y_train)):
        label = y_train[i]
        if train_counts_orig[label] > max_allowed:
            if np.random.rand() < (max_allowed / train_counts_orig[label]):
                keep_indices.append(i)
        else:
            keep_indices.append(i)
    
    X_train = X_train[keep_indices]
    y_train = y_train[keep_indices]
    print(f"After minimal balancing: {X_train.shape}")
else:
    print("\n⚠️ Dataset imbalanced - applying downsampling")
    # Original downsampling logic
    straight_idx = int(np.where(class_names == 'straight')[0]) if 'straight' in class_names else -1
    stop_idx = int(np.where(class_names == 'stop')[0]) if 'stop' in class_names else -1
    left_idx = int(np.where(class_names == 'left')[0]) if 'left' in class_names else -1
    right_idx = int(np.where(class_names == 'right')[0]) if 'right' in class_names else -1
    
    left_right_counts = [train_counts_orig.get(left_idx, 0), 
                         train_counts_orig.get(right_idx, 0)]
    left_right_counts = [c for c in left_right_counts if c > 0]
    
    if left_right_counts:
        target_count_majority = max(left_right_counts) * 3
    else:
        target_count_majority = 1000
    
    print(f"Target count for majority classes: {target_count_majority}")
    
    keep_indices = []
    for i in range(len(y_train)):
        label = y_train[i]
        if label == stop_idx or label == straight_idx:
            if np.random.rand() < (target_count_majority / max(train_counts_orig[label], 1)):
                keep_indices.append(i)
        else:
            keep_indices.append(i)
    
    X_train = X_train[keep_indices]
    y_train = y_train[keep_indices]
    print(f"After downsampling: {X_train.shape}")

# Analyze class distribution
train_counts = Counter(y_train)
print("\n=== Final class distribution ===")
for i, name in enumerate(class_names):
    count = train_counts.get(i, 0)
    orig_count = train_counts_orig.get(i, 0)
    change = count - orig_count
    print(f" - {name}: {count} (change: {change:+d})")

# Augmentation function
def augment_class(X, y, class_idx, target_n, noise_scale=0.015):
    """Augment minority class with noise"""
    idxs = np.where(y == class_idx)[0]
    if len(idxs) == 0:
        return np.empty((0,) + X.shape[1:]), np.empty((0,), dtype=y.dtype)
    need = max(0, target_n - len(idxs))
    if need == 0:
        return np.empty((0,) + X.shape[1:]), np.empty((0,), dtype=y.dtype)
    choices = np.random.choice(idxs, size=need, replace=True)
    X_extra = []
    for c in choices:
        sample = X[c].copy()
        noise = np.random.normal(loc=0.0, scale=noise_scale, size=sample.shape)
        X_extra.append(sample + noise)
    X_extra = np.array(X_extra)
    y_extra = np.full((X_extra.shape[0],), class_idx, dtype=y.dtype)
    return X_extra, y_extra

# Balance the dataset
max_count = max(train_counts.values())
print(f"\nUpsampling target per class: {max_count}")

X_train_aug = [X_train]
y_train_aug = [y_train]

for i in range(len(class_names)):
    if train_counts[i] == max_count:
        continue
    X_aug, y_aug = augment_class(X_train, y_train, i, max_count, noise_scale=0.015)
    if len(X_aug) > 0:
        X_train_aug.append(X_aug)
        y_train_aug.append(y_aug)
        print(f"Augmented class {class_names[i]}: +{len(X_aug)} samples")

X_train_np = np.concatenate(X_train_aug)
y_train_np = np.concatenate(y_train_aug)

# Shuffle
perm = np.random.permutation(len(X_train_np))
X_train_np = X_train_np[perm]
y_train_np = y_train_np[perm]

X_val_np = X_val
y_val_np = y_val
X_test_np = X_test
y_test_np = y_test

train_counts_after = Counter(y_train_np)
print("\nTraining class counts after augmentation:")
for i, name in enumerate(class_names):
    print(f" - {name}: {train_counts_after.get(i,0)}")

# Class weights for loss
total_samples = len(y_train_np)
n_classes = len(class_names)
class_weights_dict = {}
for i in range(n_classes):
    count = train_counts_after.get(i, 0)
    if count > 0:
        class_weights_dict[i] = total_samples / (n_classes * count)
    else:
        class_weights_dict[i] = 1.0

print("Class weights:", class_weights_dict)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train_np, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_np, dtype=torch.long)
X_val_tensor = torch.tensor(X_val_np, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_np, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_np, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_np, dtype=torch.long)

print("✅ Data ready for PyTorch")

✅ Data loaded: (102339, 17)
                 timestamp    epoch_time    ax_g    ay_g    az_g  gx_dps  \
0  2025-11-22_15-14-52-005  1.763805e+09  0.1689  0.4773  0.9106  -3.344   
1  2025-11-22_15-14-52-018  1.763805e+09  0.2151  0.3948  0.8076  -0.916   
2  2025-11-22_15-14-52-023  1.763805e+09  0.2100  0.3455  1.0078  -2.206   
3  2025-11-22_15-14-52-033  1.763805e+09  0.1282  0.3958  0.8982  -4.519   
4  2025-11-22_15-14-52-043  1.763805e+09  0.2749  0.4517  0.8167  -2.130   

   gy_dps  gz_dps  gps_utc     gps_lat gps_ns     gps_lon gps_ew  \
0 -12.527 -10.351  94454.0  1725.43548      N  7820.05812      E   
1  -9.092 -11.237  94454.0  1725.43548      N  7820.05812      E   
2  -7.023 -11.046  94454.0  1725.43548      N  7820.05812      E   
3 -13.153 -10.244  94454.0  1725.43548      N  7820.05812      E   
4  -8.931 -10.527  94454.0  1725.43548      N  7820.05812      E   

   gps_speed_kn  gps_course_deg gps_valid label  
0        11.568          290.39         A  stop  
1     

In [3]:
# CRITICAL: Model MUST match nov21.py EXACTLY
class BidirectionalLSTMModel(nn.Module):
    def __init__(self, input_size, hidden_sizes, num_classes, dropout_rates, l2_reg=1e-4):
        super(BidirectionalLSTMModel, self).__init__()
        self.l2_reg = l2_reg
        
        # First Bidirectional LSTM (128 units each direction = 256 total output)
        self.lstm1 = nn.LSTM(input_size, hidden_sizes[0], batch_first=True, bidirectional=True)
        self.bn1 = nn.BatchNorm1d(hidden_sizes[0] * 2)
        self.dropout1 = nn.Dropout(dropout_rates[0])
        
        # Second Bidirectional LSTM (96 units each direction = 192 total output)
        self.lstm2 = nn.LSTM(hidden_sizes[0] * 2, hidden_sizes[1], batch_first=True, bidirectional=True)
        self.bn2 = nn.BatchNorm1d(hidden_sizes[1] * 2)
        self.dropout2 = nn.Dropout(dropout_rates[1])
        
        # Dense layers
        self.fc1 = nn.Linear(hidden_sizes[1] * 2, 96)
        self.bn3 = nn.BatchNorm1d(96)
        self.dropout3 = nn.Dropout(dropout_rates[2])
        
        self.fc2 = nn.Linear(96, 48)
        self.dropout4 = nn.Dropout(dropout_rates[3])
        
        self.fc3 = nn.Linear(48, num_classes)
        
    def forward(self, x):
        # x shape: (batch, seq_len, features)
        
        # First LSTM layer
        out, _ = self.lstm1(x)
        # Take only the last timestep for batch norm
        out_last = out[:, -1, :]  # (batch, hidden*2)
        out_last = self.bn1(out_last)
        out_last = self.dropout1(out_last)
        # Expand back to sequence
        out = out_last.unsqueeze(1)  # (batch, 1, hidden*2)
        
        # Second LSTM layer
        out, _ = self.lstm2(out)
        out = out[:, -1, :]  # Take last timestep (batch, hidden*2)
        out = self.bn2(out)
        out = self.dropout2(out)
        
        # Dense layers
        out = torch.relu(self.fc1(out))
        out = self.bn3(out)
        out = self.dropout3(out)
        
        out = torch.relu(self.fc2(out))
        out = self.dropout4(out)
        
        out = self.fc3(out)
        return out
    
    def get_l2_loss(self):
        """Compute L2 regularization loss"""
        l2_loss = 0
        for name, param in self.named_parameters():
            if 'weight' in name:
                l2_loss += torch.sum(param ** 2)
        return self.l2_reg * l2_loss

# Focal Loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.reduction = reduction
        if alpha is not None:
            self.alpha = torch.tensor(alpha, dtype=torch.float32)
        else:
            self.alpha = None
    
    def forward(self, inputs, targets):
        probs = torch.softmax(inputs, dim=1)
        targets_one_hot = torch.zeros_like(probs)
        targets_one_hot.scatter_(1, targets.unsqueeze(1), 1)
        
        epsilon = 1e-7
        probs = torch.clamp(probs, epsilon, 1.0 - epsilon)
        ce = -targets_one_hot * torch.log(probs)
        
        if self.alpha is not None:
            if self.alpha.device != inputs.device:
                self.alpha = self.alpha.to(inputs.device)
            alpha_factor = targets_one_hot * self.alpha
        else:
            alpha_factor = 1.0
        
        modulating_factor = torch.pow(1.0 - probs, self.gamma)
        focal_loss = alpha_factor * modulating_factor * ce
        focal_loss = torch.sum(focal_loss, dim=1)
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

# CRITICAL: Model parameters MUST match nov21.py
model = BidirectionalLSTMModel(
    input_size=n_features,
    hidden_sizes=[128, 96],  # MATCHING nov21.py
    num_classes=n_classes,
    dropout_rates=[0.4, 0.35, 0.3, 0.25],  # MATCHING nov21.py (4 dropout layers)
    l2_reg=1e-4
).to(device)

# Compute focal loss alpha with boost for minority classes
counts = np.array([train_counts_after.get(i, 0) for i in range(n_classes)], dtype=np.float32)
counts = np.where(counts == 0, 1.0, counts)
alpha_vec = (1.0 / counts)
alpha_vec = alpha_vec / np.sum(alpha_vec)

# Extra boost for specific classes to improve diagonal
boost_classes = ['bump', 'left', 'right']
for cls_name in boost_classes:
    if cls_name in class_names:
        idx = int(np.where(class_names == cls_name)[0])
        alpha_vec[idx] *= 2.5  # Strong boost for minority classes

alpha_vec = alpha_vec / np.sum(alpha_vec)
print('Per-class alpha vector for focal loss:', alpha_vec)

criterion = FocalLoss(alpha=alpha_vec, gamma=2.0)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

# Learning rate scheduler
import inspect
_lr_kwargs = dict(mode='min', factor=0.5, patience=5, min_lr=1e-7)
if 'verbose' in inspect.signature(optim.lr_scheduler.ReduceLROnPlateau).parameters:
    _lr_kwargs['verbose'] = True
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, **_lr_kwargs)

# Class weights tensor
class_weights_tensor = torch.tensor([class_weights_dict[i] for i in range(n_classes)], dtype=torch.float32)

print("\n✅ Model Summary:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Per-class alpha vector for focal loss: [0.26315787 0.26315787 0.26315787 0.10526315 0.10526315]

✅ Model Summary:
BidirectionalLSTMModel(
  (lstm1): LSTM(7, 128, batch_first=True, bidirectional=True)
  (bn1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout1): Dropout(p=0.4, inplace=False)
  (lstm2): LSTM(256, 96, batch_first=True, bidirectional=True)
  (bn2): BatchNorm1d(192, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout2): Dropout(p=0.35, inplace=False)
  (fc1): Linear(in_features=192, out_features=96, bias=True)
  (bn3): BatchNorm1d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout3): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=96, out_features=48, bias=True)
  (dropout4): Dropout(p=0.25, inplace=False)
  (fc3): Linear(in_features=48, out_features=5, bias=True)
)

Total parameters: 436,677
Trainable parameters: 436,677

✅ Model Summary:
BidirectionalLSTMModel(
  (lstm1)

In [4]:
# Prepare PyTorch datasets and dataloaders
train_dataset = TensorDataset(
    torch.from_numpy(X_train_np).float(),  # Ensure float32
    torch.from_numpy(y_train_np).long()
)
val_dataset = TensorDataset(
    torch.from_numpy(X_val_np).float(),  # Ensure float32
    torch.from_numpy(y_val_np).long()
)
test_dataset = TensorDataset(
    torch.from_numpy(X_test_np).float(),  # Ensure float32
    torch.from_numpy(y_test_np).long()
)

# Batch size for training
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")
print(f"✅ Data loaders ready (dtype: float32)")

Train batches: 25
Val batches: 6
Test batches: 6
✅ Data loaders ready (dtype: float32)


In [ ]:
# Training function
def train_epoch(model, train_loader, criterion, optimizer, device, class_weights):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        
        loss = criterion(outputs, labels)
        loss = loss + model.get_l2_loss()
        
        weights = class_weights[labels].to(device)
        loss = (loss * weights).mean()
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

# Validation function
def validate(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss = loss + model.get_l2_loss()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc, np.array(all_preds), np.array(all_labels)

# Per-class metrics
def compute_per_class_metrics(y_true, y_pred, class_names):
    from sklearn.metrics import recall_score, confusion_matrix
    recalls = recall_score(y_true, y_pred, labels=np.arange(len(class_names)), average=None, zero_division=0)
    
    # Compute diagonal (per-class accuracy)
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
    per_class_acc = cm.diagonal() / cm.sum(axis=1).clip(min=1)
    
    print("  Per-class metrics:")
    for i, name in enumerate(class_names):
        print(f"    {name.upper():10s} - Recall: {recalls[i]:.4f}, Diagonal: {per_class_acc[i]:.4f}")
    
    return recalls, per_class_acc

# Training loop - INCREASED patience and epochs
num_epochs = 80  # More epochs for better convergence
patience = 20  # More patience to find best model
best_val_loss = float('inf')
patience_counter = 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print("Starting training...")
print("="*70)
for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, class_weights_tensor)
    val_loss, val_acc, val_preds, val_labels = validate(model, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Epoch {epoch+1:02d}/{num_epochs} - "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} - "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    recalls, diagonals = compute_per_class_metrics(val_labels, val_preds, class_names)
    
    scheduler.step(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        
        # Save checkpoint with ALL required info for nov21.py
        torch.save({
            'model_state_dict': model.state_dict(),
            'scaler_mean': scaler.mean_.tolist(),
            'scaler_scale': scaler.scale_.tolist(),
            'classes': label_encoder.classes_.tolist()
        }, "best_model.pth")
        
        print("  ✔ Saved best_model.pth")
        
        # Check if all diagonals > 0.6
        if np.all(diagonals >= 0.6):
            print(f"  🎯 TARGET ACHIEVED! All diagonal values >= 0.6")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered after {epoch+1} epochs")
            break
    print("-"*70)

# Load best model
checkpoint = torch.load('best_model.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print("\n✅ Training complete! Best model restored.")
print(f"✅ Model saved to: best_model.pth")
print(f"✅ Classes: {checkpoint['classes']}")
print(f"✅ Window size: {n_time_steps}, Features: {n_features}")

Starting training...
Epoch 01/80 - Train Loss: 0.2795, Train Acc: 0.6436 - Val Loss: 0.2414, Val Acc: 0.4726
  Per-class metrics:
    BUMP       - Recall: 0.0000, Diagonal: 0.0000
    LEFT       - Recall: 0.6897, Diagonal: 0.6897
    RIGHT      - Recall: 0.8161, Diagonal: 0.8161
    STOP       - Recall: 0.9938, Diagonal: 0.9938
    STRAIGHT   - Recall: 0.1656, Diagonal: 0.1656
  ✔ Saved best_model.pth
----------------------------------------------------------------------
Epoch 01/80 - Train Loss: 0.2795, Train Acc: 0.6436 - Val Loss: 0.2414, Val Acc: 0.4726
  Per-class metrics:
    BUMP       - Recall: 0.0000, Diagonal: 0.0000
    LEFT       - Recall: 0.6897, Diagonal: 0.6897
    RIGHT      - Recall: 0.8161, Diagonal: 0.8161
    STOP       - Recall: 0.9938, Diagonal: 0.9938
    STRAIGHT   - Recall: 0.1656, Diagonal: 0.1656
  ✔ Saved best_model.pth
----------------------------------------------------------------------
Epoch 02/80 - Train Loss: 0.1909, Train Acc: 0.7896 - Val Loss: 0.170

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "scaler_mean": scaler.mean_.tolist(),
    "scaler_scale": scaler.scale_.tolist(),
    "classes": label_encoder.classes_.tolist(),
}, "imu_classifier_model_clean.pth")


In [ ]:
# 🔄 RELOAD BEST MODEL (run this if you interrupted training)
# This ensures evaluation uses the best saved checkpoint, not the in-memory model

import os
if os.path.exists('best_model.pth'):
    checkpoint = torch.load('best_model.pth', map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print("✅ Loaded best model from best_model.pth")
    print(f"   Scaler mean: {checkpoint['scaler_mean'][:3]}... (first 3 values)")
    print(f"   Classes: {checkpoint['classes']}")
else:
    print("⚠️ No checkpoint found! Please run the training cell first.")


In [ ]:
# Evaluate on validation and test sets
_, val_acc, _, _ = validate(model, val_loader, criterion, device)
_, test_acc, _, _ = validate(model, test_loader, criterion, device)

print(f"✅ Validation Accuracy: {val_acc:.4f}")
print(f"✅ Test Accuracy: {test_acc:.4f}")


In [ ]:
# Get the class labels
classes = label_encoder.classes_

from sklearn.metrics import confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

# Get predictions on test set
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

y_pred_classes = np.array(all_preds)
y_test_categorical = np.array(all_labels)

print("Shape of y_pred_classes is", y_pred_classes.shape)
print("Shape of y_test_categorical is", y_test_categorical.shape)

# Ensure confusion_matrix uses the full set of labels in the correct order
conf_matrix = confusion_matrix(y_test_categorical, y_pred_classes, labels=np.arange(len(classes)))

# Calculate accuracy
accuracy = accuracy_score(y_test_categorical, y_pred_classes)

# Normalize the confusion matrix by row (handle possible zero-row sums)
row_sums = conf_matrix.sum(axis=1, keepdims=True)
conf_matrix_normalized = conf_matrix.astype('float') / np.where(row_sums == 0, 1, row_sums)

# Plot the confusion matrix using seaborn with readable label names
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix_normalized, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=classes, yticklabels=classes, cbar=True)
plt.title('Normalized Confusion Matrix for Test Dataset\nAccuracy: {:.2%}'.format(accuracy))
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print(f"Accuracy: {accuracy * 100:0.8f}%")


In [ ]:
# Get the class labels
classes = label_encoder.classes_

from sklearn.metrics import confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

# Get predictions on training set
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

y_pred_classes = np.array(all_preds)
y_train_categorical = np.array(all_labels)

print("Shape of y_pred_classes is", y_pred_classes.shape)
print("Shape of y_train_categorical is", y_train_categorical.shape)

# Ensure confusion_matrix uses the full set of labels in the correct order
conf_matrix = confusion_matrix(y_train_categorical, y_pred_classes, labels=np.arange(len(classes)))

# Calculate accuracy
accuracy = accuracy_score(y_train_categorical, y_pred_classes)

# Normalize the confusion matrix by row (handle possible zero-row sums)
row_sums = conf_matrix.sum(axis=1, keepdims=True)
conf_matrix_normalized = conf_matrix.astype('float') / np.where(row_sums == 0, 1, row_sums)

# Plot the confusion matrix using seaborn with readable label names
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix_normalized, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=classes, yticklabels=classes, cbar=True)
plt.title('Normalized Confusion Matrix for Train Dataset\nAccuracy: {:.2%}'.format(accuracy))
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print(f"Accuracy: {accuracy * 100:0.8f}%")


In [ ]:
if 'history' in locals() and len(history['train_loss']) > 0:
    plt.figure(figsize=(8,5))
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.title('Training vs Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    plt.figure(figsize=(8,5))
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Val Accuracy')
    plt.title('Training vs Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("⚠️ Please run the model training cell (cell 5) first to generate the history object.")
